In [ ]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
import pyarrow.dataset as ds
import json

food_paquet = "../../data/food.parquet"

# import os
# print(os.getcwd())
# print(os.path.exists("data/food.parquet"))

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# pq.read_pandas('data/food.parquet', columns=['lang']).to_pandas()
# pq.read_pandas('data/food.parquet', columns=['lang'], filters=[('lang', '==', 'fr')]).to_pandas()

# pq.read_pandas('data/food.parquet', columns=['nutriments']).to_pandas().head(5)

#pq.read_schema('data/food.parquet').names
#pq.read_schema('data/food.parquet')
# pq.read_schema('data/food.parquet').field('nutriments').type

# open_food_facts_df = pd.read_parquet(food_paquet, columns=("countries_tags"))
# open_food_facts_df = pq.read_table(food_paquet, columns=["countries_tags", "nutriments"])

# colonnes = open_food_facts_df.column_names
# type_colonnes = open_food_facts_df.field("nutriments").type

# print(colonnes)
# print(type_colonnes)

# print(open_food_facts_df)

# non utilisé on va faire plutot des fonctions par type de colonnes qu'on a au préalable choisis
# def save_set_unique(col: str):

#     ar_col = pq.read_table(food_paquet, columns=[col])
#     t = ar_col.schema.field(col).type
#     val_list = ar_col.to_pylist()

#     result = set()

#     if pa.types.is_list(t) or pa.types.is_large_list(t):
#         for line in val_list:
#             if line is not None:
#                 for element in line:
#                     if element is not None:
#                         if isinstance(element, dict): 
#                             for valeur in element:
#                                 result.add(valeur)
#                         else:
#                             result.add(element)                    
#         return result


# Fonction récursive qui affiche le type et structure d'un champ pyarrow

def describe_field(field: pa.Field, indent: int = 0):

    prefix = "  " * indent
    t = field.type

    if pa.types.is_list(t) or pa.types.is_large_list(t):
        print(f"{prefix}{field.name}: LIST de")
        describe_field(t.value_field, indent + 1)

    elif pa.types.is_struct(t):
        print(f"{prefix}{field.name}: STRUCT avec les champs:")
        for sub_field in t:
            describe_field(sub_field, indent + 1)

    else:
        print(f"{prefix}{field.name}: {t}")

# Parcourt toutes les colonnes d'un fichier parquet et affiche leur type et structure grace à describe field
def describe_schema(path: str):
    schema = pq.read_schema(path)
    print(f"Fichier: {path} — {len(schema)} colonnes\n")
    for field in schema:
        describe_field(field)
        print()

describe_schema(food_paquet)

In [ ]:

def verif_col_string(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    for valeur in val_list:
        # print(type(valeur[col]))
        if valeur[col] is None:
            null_compt += 1
        # if not isinstance(valeur[col], str):
        #     print(valeur[col])
        #     type_compt += 1
    # print(len(val_list))
    return null_compt

verif_col_string("brands")

In [ ]:
def verif_col_codeb(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    false_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        if len(valeur[col]) > 13 or len(valeur[col]) < 13:
            false_compt += 1
        # if not isinstance(valeur[col], str):
        #     print(valeur[col])
    # print(len(val_list))
    return null_compt, false_compt

null_compt, false_compt = verif_col_codeb("code")
print(f"valeur nulle : {null_compt} valeur fausse : {false_compt}")

In [ ]:
def verif_col_complet(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    false_compt = 0
    for valeur in val_list:
        # print(type(valeur[col]))
        if valeur[col] is None:
            null_compt += 1
        if valeur[col] is not None and (valeur[col] > 1 or valeur[col] == 0):
            false_compt += 1
    # print(len(val_list))
    return null_compt, false_compt

null_compt, typerr_compt = verif_col_complet("completeness")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt}")

In [ ]:
def verif_col_countrytag(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        else:
            for item in valeur[col]:
                # print(type(item))
                if not isinstance(item, str):
                    typerr_compt += 1
    # print(len(val_list))
    return null_compt, typerr_compt

null_compt, typerr_compt = verif_col_countrytag("countries_tags")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt}")

In [ ]:
def verif_col_data_quality(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        else:
            for item in valeur[col]:
                # print(type(item))
                if not isinstance(item, str):
                    typerr_compt += 1
    # print(len(val_list))
    return null_compt, typerr_compt

null_compt, typerr_compt = verif_col_data_quality("data_quality_errors_tags")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt}")

In [ ]:
import re

def is_date_valide(date_str):
    # passage au regex division par 4 du temps d'éxécution
    return bool(re.match(r"^\d{4}(-\d{2}(-\d{2})?)?$", date_str))
    # formats = ["%Y-%m-%d", "%Y-%m", "%Y"]
    # for fmt in formats:
    #     try:
    #         datetime.strptime(date_str, fmt)
    #         return True
    #     except ValueError:
    #         continue
    # return False

def verif_col_entry_dates(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    daterr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        else:
            for item in valeur[col]:
                if not isinstance(item, str):
                    typerr_compt += 1
                if not is_date_valide(item):
                    daterr_compt += 1
    # print(len(val_list))
    return null_compt, typerr_compt, daterr_compt

null_compt, typerr_compt, daterr_compt = verif_col_entry_dates("entry_dates_tags")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt} erreur de date : {daterr_compt}")

In [ ]:
def verif_col_food_groups(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
        else:
            for item in valeur[col]:
                # print(type(item))
                if not isinstance(item, str):
                    typerr_compt += 1
    # print(len(val_list))
    return null_compt, typerr_compt

null_compt, typerr_compt = verif_col_data_quality("food_groups_tags")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt}")

In [ ]:
def verif_col_analysys(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    empty_compt = 0
    for valeur in val_list:
        # print(valeur[col])
        if valeur[col] is None:
            null_compt += 1
        elif valeur[col] == []:
            empty_compt += 1
        else:
            for item in valeur[col]:
                # print(item)
                if not isinstance(item, str):
                    typerr_compt += 1
    # print(len(val_list))
    return null_compt, typerr_compt, empty_compt

null_compt, typerr_compt, empty_compt = verif_col_analysys("ingredients_analysis_tags")
print(f"valeur nulle : {null_compt} erreur de type : {typerr_compt} valeur vide : {empty_compt}")

In [ ]:
def verif_col_int(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    type_compt = 0
    for valeur in val_list:
        # print(type(valeur[col]))
        if valeur[col] is None:
            null_compt += 1
        elif not isinstance(valeur[col], int):
        #     print(valeur[col])
            type_compt += 1
    # print(len(val_list))
    return null_compt, type_compt

verif_col_int("nutriscore_score")

In [ ]:
def check_col_string(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    for valeur in val_list:
        # print(type(valeur[col]))
        if valeur[col] is None:
            null_compt += 1
        # if not isinstance(valeur[col], str):
        #     print(valeur[col])
        #     type_compt += 1
    nb_ligne = len(val_list)
    percent_null = round((null_compt / nb_ligne) * 100, 2)
    return null_compt, percent_null

check_col_string("ingredients")

In [ ]:
def check_col_liststr(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
    nb_ligne = len(val_list)
    percent_null = round((null_compt / nb_ligne) * 100, 4)
    return null_compt, percent_null

check_col_liststr("ingredients_text")

In [ ]:
# ne pas lancer dépasse les 10 min d'éxécution

def check_col_liststr(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
    nb_ligne = len(val_list)
    percent_null = round((null_compt / nb_ligne) * 100, 4)
    return null_compt, percent_null

check_col_liststr("nutriments")

In [ ]:
def check_col_liststr(col: str):
    ar_col = pq.read_table(food_paquet, columns=[col])
    val_list = ar_col.to_pylist()
    null_compt = 0
    typerr_compt = 0
    for valeur in val_list:
        if valeur[col] is None:
            null_compt += 1
    nb_ligne = len(val_list)
    percent_null = round((null_compt / nb_ligne) * 100, 4)
    return null_compt, percent_null

check_col_liststr("product_name")

Rappel ce qu'on utilise dans les fonctions suivantes pour accélerer l'éxécution : 

chunk : un fichier Parquet est lu par pandas/pyarrow en plusieurs morceaux (chunks) pour des raisons de performance/mémoire — la colonne nutriments n'est pas un seul bloc mais une liste de plusieurs ListArray (les chunks). chunk.flatten() déplie chaque morceau individuellement (transforme la liste de structs en un seul tableau de structs à plat).
concat_arrays : après avoir flatten chaque chunk séparément, tu obtiens plusieurs tableaux "aplatis" distincts (un par chunk d'origine). pa.concat_arrays() les recolle en un seul grand tableau, pour pouvoir faire tes calculs/filtres sur l'ensemble des données d'un coup, plutôt que de gérer plusieurs morceaux séparés.

En résumé : on traite chunk par chunk (obligatoire techniquement), puis on rassemble le tout pour analyser globalement.

In [ ]:
# procedure va verifier les null dans l'un des champs d'un dictionnaire (field) d'une colonne (col) et retourner le nombre d'occurence et le pourcentage

def check_col_liststr(col: str, field: str = "name"):
    table = pq.read_table(food_paquet, columns=[col])
    chunked = table.column(col)  # ChunkedArray de type list<struct<...>>

    null_compt = 0
    total_elements = 0

    for chunk in chunked.chunks:
        # flatten() "déplie" la liste : on obtient un StructArray
        # contenant tous les éléments de toutes les lignes de ce chunk
        flat_struct = chunk.flatten()
        field_arr = flat_struct.field(field)

        null_compt += pc.sum(pc.is_null(field_arr)).as_py() or 0
        total_elements += len(field_arr)

    percent_null = round((null_compt / total_elements) * 100, 4) if total_elements else 0.0
    return null_compt, percent_null

check_col_liststr("nutriments", "name")

In [ ]:
def check_col_liststr_par_ligne(col: str, field: str = "name"):
    table = pq.read_table(food_paquet, columns=[col])
    chunked = table.column(col)

    lignes_affectees = 0
    total_lignes = 0

    for chunk in chunked.chunks:
        offsets = chunk.offsets.to_numpy(zero_copy_only=False)
        flat_struct = chunk.flatten()
        field_arr = flat_struct.field(field)

        is_null = pc.is_null(field_arr).to_numpy(zero_copy_only=False)

        # Pour chaque ligne, on regarde si au moins un élément entre
        # offsets[i] et offsets[i+1] est null
        for i in range(len(offsets) - 1):
            start, end = offsets[i], offsets[i + 1]
            if start == end:
                continue  # liste vide, rien à vérifier
            if is_null[start:end].any():
                lignes_affectees += 1

        total_lignes += len(offsets) - 1

    percent = round((lignes_affectees / total_lignes) * 100, 4) if total_lignes else 0.0
    return lignes_affectees, percent

check_col_liststr_par_ligne("nutriments", "100g")

In [ ]:
#procedure retourne le nombre d'occurence par unité

from collections import Counter

def get_units_with_counts(col: str = "nutriments", field: str = "unit"):
    table = pq.read_table(food_paquet, columns=[col])
    chunked = table.column(col)

    counter = Counter()

    for chunk in chunked.chunks:
        flat_struct = chunk.flatten()
        field_arr = flat_struct.field(field)
        vc = pc.value_counts(field_arr)  # struct array {values, counts}
        for entry in vc.to_pylist():
            counter[entry["values"]] += entry["counts"]

    return counter.most_common()

get_units_with_counts("nutriments", "unit")

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

def get_field_stats_by_name_unit(col: str = "nutriments", value_field: str = "100g"):
    table = pq.read_table(food_paquet, columns=[col])
    chunked = table.column(col)

    flat_arrays = [chunk.flatten() for chunk in chunked.chunks]
    flat_struct = pa.concat_arrays(flat_arrays)

    names = flat_struct.field("name")
    units = flat_struct.field("unit")
    values = flat_struct.field(value_field)

    tmp_table = pa.table({"name": names, "unit": units, value_field: values})

    # result = tmp_table.group_by(["name", "unit"]).aggregate([
    result = tmp_table.group_by(["unit"]).aggregate([
        (value_field, "min"),
        (value_field, "max"),
        (value_field, "mean"),
        (value_field, "count"),
    ])

    return result.to_pandas().sort_values(f"{value_field}_count", ascending=False)


get_field_stats_by_name_unit("nutriments", "100g")

In [ ]:
#Utilisation des percentiles pour représenter a quoi ressemble la donnée typique (plus utile qu'une moyenne ici)
#un percentile indique la valeur en dessous (ou égal) de laquelle se retrouve un certain pourcentage de données
# la colonne unit percent est le % total de nutriment utilisant l'unité indiqué

def inspect_all_units(col: str = "nutriments", value_field: str = "100g"):
    table = pq.read_table(food_paquet, columns=[col])
    chunked = table.column(col)

    flat_arrays = [chunk.flatten() for chunk in chunked.chunks]
    flat_struct = pa.concat_arrays(flat_arrays)

    units_arr = flat_struct.field("unit")
    values_arr = flat_struct.field(value_field)

    tmp_table = pa.table({"unit": units_arr, value_field: values_arr})

    df = tmp_table.to_pandas()

    results = df.groupby("unit", dropna=False)[value_field].describe(
        percentiles=[.01, .05, .25, .5, .75, .95, .99, .999]
    ).drop(columns=["mean", "std", "min", "max"]).sort_values("count", ascending=False)

    results["unit_percent"] = round((results["count"] / results["count"].sum()) * 100, 4)

    return results  # transposé : une ligne par unité


inspect_all_units("nutriments", "100g")

ok si l'on veut convertir uniquement les mg et autre en g il faut cibler les unités qui sont compatible, exit les kj et kcal qui représent l'énergie des nutriments

In [ ]:
table = pq.read_table(food_paquet, columns=["nutriments"])
chunked = table.column("nutriments")
flat_arrays = [chunk.flatten() for chunk in chunked.chunks]
flat_struct = pa.concat_arrays(flat_arrays)

mask = pc.equal(flat_struct.field("unit"), "")
filtered_entries = flat_struct.filter(mask)

filtered_entries.to_pandas().head(60000)

In [ ]:
table = pq.read_table(food_paquet, columns=["nutriments"])
chunked = table.column("nutriments")
flat_arrays = [chunk.flatten() for chunk in chunked.chunks]
flat_struct = pa.concat_arrays(flat_arrays)

mask = pc.is_null(flat_struct.field("unit"))
filtered_entries = flat_struct.filter(mask)

# filtered_entries.to_pandas().head(1800000)

vc = pc.value_counts(filtered_entries.field("name"))

name_counts = pd.DataFrame({
    "name": vc.field("values").to_pandas(),
    "count": vc.field("counts").to_pandas()
})

name_counts = name_counts.sort_values("count", ascending=False).reset_index(drop=True)

name_counts

on répertorie les différentes valeurs de nom de nutrimet qui n'ont pas d'unité précisé